# Phase 3: NLP Signal Extraction

Turns the raw review text in `reviews_clean.csv` into structured numeric
features that later phases (segmentation, risk classification, growth) can
use directly.

Two kinds of signal:
1. **Sentiment** — how positive/negative each review reads, via VADER
   (lexicon-based, no training needed, well-suited to short informal text
   like reviews).
2. **Structural text features** — simple counts (length, punctuation) that
   are cheap to compute and often carry real signal (e.g. very short,
   all-caps, exclamation-heavy reviews tend to correlate with strong
   sentiment in either direction).

**Not doing NER/event tagging** for this dataset — see
`docs/capstone-plan-of-action.md` Phase 3 for why: Yelp reviews describe a
single customer visit, not business events (hiring, expansion, etc.), so
generic entity extraction wouldn't produce meaningful signal for the
risk/segmentation/growth tasks here. This is a data-source-driven scope
decision, not a shortcut — noted explicitly for the Phase 7 write-up.

## Load the cleaned reviews

In [1]:
import pandas as pd

reviews = pd.read_csv('../data/reviews_clean.csv', parse_dates=['date'])
reviews.shape

(967489, 9)

## Sentiment scoring (VADER)

VADER's `polarity_scores()` returns four numbers per piece of text: `neg`,
`neu`, `pos` (proportions that sum to 1) and `compound` — a single
normalized score from -1 (most negative) to +1 (most positive). `compound`
is the one most useful as a single summary feature downstream, but we'll
keep all four in case `neg`/`pos` prove useful separately (e.g. a review
that's simultaneously very positive and very negative in different parts —
mixed praise/complaint — would show up as high in both, which `compound`
alone would mask).

With ~967K reviews, running this row-by-row with `.apply()` is fine — VADER
is a lexicon lookup, not a model inference call, so it's fast (roughly
1-2 minutes for the full dataset).

In [2]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

sentiment_scores = reviews['text'].apply(sia.polarity_scores).apply(pd.Series)
reviews[['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound']] = sentiment_scores[['neg', 'neu', 'pos', 'compound']]

reviews[['text', 'stars', 'sentiment_compound']].head()

,text,stars,sentiment_compound
0,I've taken a lot of spin classes over the year...,5,0.9858
1,"Wow! Yummy, different, delicious. Our favo...",5,0.9588
2,I am a long term frequent customer of this est...,1,0.7117
3,Good food--loved the gnocchi with marinara\nth...,4,0.8093
4,Tremendous service (Big shout out to Douglas) ...,5,0.8360


## Structural text features

Cheap-to-compute properties of the raw text itself, independent of sentiment:

- `text_word_count` — very short or very long reviews may behave differently
- `text_exclamation_count` — enthusiasm/emphasis marker
- `text_question_count` — often signals confusion or a complaint framed as a
  question ("why would they...")
- `text_caps_word_ratio` — share of words that are ALL CAPS (3+ letters, to
  avoid false positives like "I" or "A"), another emphasis/emotion marker

In [3]:
def caps_word_ratio(text):
    words = text.split()
    if not words:
        return 0.0
    caps_words = [w for w in words if len(w) >= 3 and w.isupper()]
    return len(caps_words) / len(words)

reviews['text_word_count'] = reviews['text'].str.split().str.len()
reviews['text_exclamation_count'] = reviews['text'].str.count('!')
reviews['text_question_count'] = reviews['text'].str.count(r'\?')
reviews['text_caps_word_ratio'] = reviews['text'].apply(caps_word_ratio)

reviews[['text_word_count', 'text_exclamation_count', 'text_question_count', 'text_caps_word_ratio']].describe()

,text_word_count,text_exclamation_count,text_question_count,text_caps_word_ratio
count,967489.000000,967489.000000,967489.000000,967489.000000
mean,112.634656,1.206120,0.193081,0.005243
std,100.279304,2.405409,0.718048,0.015932
min,1.000000,0.000000,0.000000,0.000000
25%,46.000000,0.000000,0.000000,0.000000
50%,83.000000,0.000000,0.000000,0.000000
75%,145.000000,2.000000,0.000000,0.003460
max,1031.000000,489.000000,42.000000,1.000000


## Sanity-check by hand

Two checks, per the plan doc's Phase 3 checklist:

1. Does `sentiment_compound` broadly agree with the review's own star
   rating? It shouldn't match perfectly (a 3-star review is often mixed
   praise/complaint, and sentiment ≠ rating), but 1-star reviews should
   skew negative and 5-star should skew positive on average.
2. Read a handful of actual review texts next to their scores and confirm
   they look sensible — this is the "sanity-check outputs on a handful of
   rows by hand" step, done for real rather than skipped.

In [4]:
reviews.groupby('stars')['sentiment_compound'].mean()

stars
1   -0.196075
2    0.284258
3    0.644768
4    0.846830
5    0.874992
Name: sentiment_compound, dtype: float64

In [5]:
pd.set_option('display.max_colwidth', 200)

sample = reviews.sample(5, random_state=42)
sample[['stars', 'sentiment_compound', 'text']]

,stars,sentiment_compound,text
267777,4,0.8531,"The seafood is fresh which is a big pro. The con is that the ""rolls"" are the size of hotdog buns. Not worth it in my opinion. If you're feeling like dropping $$ on an eatery like this I highly sug..."
324338,4,0.8669,"Really nice neighborhood cafe/restaurant. Drop in here for coffee and a delicious piece of baklava, or a casual dinner. The menu is simple, but really wonderful."
737097,4,0.9207,"Pizza is very good. It's the best I've found in South Philly. I usually do pick up so I can't comment on their delivery service, but they are usually on point and quick for their pick up orders. \..."
116160,4,0.9365,Went with my coworkers. Awesome selection of beers and sours. I had the Raspberry Ale which was good but a little too tart for me as well as another - I believe it was called White Ale very ligh...
700755,1,0.9835,Wild horses couldn't drag me back to Waterworks. We had a reservation for 7pm. We were eventually seated around 8. It's irritating enough to be forced to wait for what turned out to be mediocre fo...


## Save output

Writes the reviews table back out with the new NLP feature columns added,
ready for Phase 4 (feature engineering) to merge with the structured
business data.

In [6]:
reviews.to_csv('../data/reviews_with_sentiment.csv', index=False)
reviews.shape

(967489, 17)